# EquiTwin Pune/PMPML — Data Visualization Notebook
**Member 4 | Analytics, Visualization & IEEE Documentation Lead**  
Dataset Year: 2025 (simulation baseline) | Random Seed: 42

All charts in this notebook are saved to `./charts/` for use in the IEEE paper.

---


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import os, warnings
warnings.filterwarnings('ignore')

# ── Output directory ─────────────────────────────────────────
os.makedirs("charts", exist_ok=True)

# ── Paths ────────────────────────────────────────────────────
DATA_DIR   = "."
TRAFFIC_F  = f"{DATA_DIR}/traffic_conditions.csv"
INCIDENTS_F= f"{DATA_DIR}/incidents.csv"
ZONES_F    = f"{DATA_DIR}/zones.csv"
ROUTES_F   = f"{DATA_DIR}/routes.txt"
STOPS_F    = f"{DATA_DIR}/stops.txt"
TRIPS_F    = f"{DATA_DIR}/trips.txt"
STOP_TIMES_F = f"{DATA_DIR}/stop_times.txt"

np.random.seed(42)

# ── Style ────────────────────────────────────────────────────
DARK_BG   = '#0f1117'
SURFACE   = '#1a1d27'
BORDER    = '#2e3354'
ACCENT    = '#6c63ff'
RED       = '#ff6b6b'
GREEN     = '#43e97b'
ORANGE    = '#f7971e'
YELLOW    = '#ffd93d'
MUTED     = '#8b90b3'
TEXT      = '#e8eaf6'

plt.rcParams.update({
    'figure.facecolor' : DARK_BG,
    'axes.facecolor'   : SURFACE,
    'axes.edgecolor'   : BORDER,
    'axes.labelcolor'  : TEXT,
    'xtick.color'      : MUTED,
    'ytick.color'      : MUTED,
    'text.color'       : TEXT,
    'grid.color'       : BORDER,
    'grid.alpha'       : 0.5,
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.titlepad'    : 12,
    'legend.facecolor' : SURFACE,
    'legend.edgecolor' : BORDER,
    'legend.labelcolor': TEXT,
    'savefig.dpi'      : 150,
    'savefig.bbox'     : 'tight',
    'savefig.facecolor': DARK_BG,
})

SLOT_ORDER = ['night_00_06','early_am_06_07','peak_am_07_10',
              'midday_10_16','peak_pm_17_20','evening_20_23','late_night_23_24']
SLOT_LABELS = ['Night\n00–06','Early AM\n06–07','Peak AM\n07–10',
               'Midday\n10–16','Peak PM\n17–20','Evening\n20–23','Late Night\n23–24']

print("✓ Libraries loaded | charts/ directory ready")


In [ ]:
traffic   = pd.read_csv(TRAFFIC_F)
incidents = pd.read_csv(INCIDENTS_F)
zones     = pd.read_csv(ZONES_F)
routes    = pd.read_csv(ROUTES_F)
stops     = pd.read_csv(STOPS_F)
trips     = pd.read_csv(TRIPS_F)
stop_times= pd.read_csv(STOP_TIMES_F)

# Derive waiting time estimate if reliability_score present
if 'reliability_score' in zones.columns and 'transport_dependency' in zones.columns:
    mean_delay = traffic['delay_multiplier'].mean()
    mean_inc   = incidents['duration_minutes'].mean() / 60
    zones['est_wait_peak_min'] = (
        (10 / (zones['reliability_score'] / 100)) * mean_delay +
        mean_inc * 60 * zones['transport_dependency'] * 0.15
    ).round(1)

print("Datasets loaded successfully.")
print(f"  traffic: {traffic.shape}  |  incidents: {incidents.shape}  |  zones: {zones.shape}")


## 2. Traffic — Speed Profile Across Time Slots

In [ ]:
# Fig 1: Average speed across 7 time slots, split by route type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig 1  |  Speed Profile Across Time Slots', color=TEXT, fontsize=13, y=1.02)

slot_speed = traffic.groupby(['time_slot','route_type'])['average_speed_kmh'].mean().unstack()
slot_speed = slot_speed.reindex(SLOT_ORDER)

ax = axes[0]
ax.plot(SLOT_LABELS, slot_speed.get('BRTS', []), marker='o', color=GREEN, linewidth=2.5,
        label='BRTS', markersize=7, markerfacecolor=GREEN)
ax.plot(SLOT_LABELS, slot_speed.get('Non-BRTS', []), marker='s', color=RED, linewidth=2.5,
        label='Non-BRTS', markersize=7, markerfacecolor=RED)
ax.axhspan(0, 16, alpha=0.07, color=RED, label='Critical zone (<16 km/h)')
ax.set_ylabel('Average Speed (km/h)', color=TEXT)
ax.set_title('BRTS vs Non-BRTS Speed by Slot')
ax.legend()
ax.grid(axis='y')
ax.set_ylim(8, 32)
ax.tick_params(axis='x', rotation=30)

# Overall city average with fill
ax2 = axes[1]
city_avg = traffic.groupby('time_slot')['average_speed_kmh'].mean().reindex(SLOT_ORDER)
city_std = traffic.groupby('time_slot')['average_speed_kmh'].std().reindex(SLOT_ORDER)
ax2.plot(SLOT_LABELS, city_avg.values, marker='D', color=ACCENT, linewidth=2.5, markersize=7)
ax2.fill_between(SLOT_LABELS,
                 (city_avg - city_std).values,
                 (city_avg + city_std).values,
                 alpha=0.15, color=ACCENT, label='±1 std dev')
ax2.axhline(19, linestyle='--', color=YELLOW, linewidth=1.5, label='TomTom city avg (19 km/h)')
ax2.set_ylabel('Speed (km/h)', color=TEXT)
ax2.set_title('City-Wide Speed with Variability Band')
ax2.legend(fontsize=9)
ax2.grid(axis='y')
ax2.set_ylim(8, 32)
ax2.tick_params(axis='x', rotation=30)

for ax in axes:
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values(): spine.set_edgecolor(BORDER)

plt.tight_layout()
plt.savefig('charts/fig1_speed_profile.png')
plt.show()
print("Saved: charts/fig1_speed_profile.png")


## 3. Congestion Heatmap — Route × Time Slot

In [ ]:
# Fig 2: Pivot heatmap of congestion level (%)
pivot = traffic.pivot(index='route_id', columns='time_slot', values='congestion_level')
pivot = pivot[SLOT_ORDER]

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(DARK_BG)

cmap = sns.diverging_palette(145, 10, s=80, l=40, as_cmap=True)
sns.heatmap(pivot, ax=ax, cmap=cmap, annot=True, fmt='.0f',
            linewidths=0.5, linecolor=DARK_BG,
            cbar_kws={'label': 'Congestion Level (%)', 'shrink': 0.8},
            annot_kws={'size': 9, 'color': TEXT})

ax.set_title('Fig 2  |  Congestion Heatmap — Route × Time Slot
(TomTom delay index: >100% = trip takes >2× free-flow time)',
             color=TEXT, pad=14)
ax.set_xlabel('Time Slot', color=TEXT, labelpad=10)
ax.set_ylabel('Route ID', color=TEXT, labelpad=10)
ax.set_xticklabels(SLOT_LABELS, rotation=30, ha='right', color=TEXT)
ax.set_yticklabels(ax.get_yticklabels(), color=TEXT)
ax.collections[0].colorbar.ax.yaxis.label.set_color(TEXT)
ax.collections[0].colorbar.ax.tick_params(colors=TEXT)

plt.tight_layout()
plt.savefig('charts/fig2_congestion_heatmap.png')
plt.show()
print("Saved: charts/fig2_congestion_heatmap.png")


## 4. Delay Multiplier by Time Slot

In [ ]:
# Fig 3: Bar chart of mean delay multiplier per slot
delay_slot = traffic.groupby('time_slot')['delay_multiplier'].agg(['mean','std']).reindex(SLOT_ORDER)

colors = [GREEN if m < 1.2 else ORANGE if m < 1.7 else RED for m in delay_slot['mean']]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(SLOT_LABELS, delay_slot['mean'], color=colors, edgecolor=BORDER,
              linewidth=1.2, width=0.6, zorder=3)
ax.errorbar(SLOT_LABELS, delay_slot['mean'], yerr=delay_slot['std'],
            fmt='none', color=TEXT, capsize=5, linewidth=1.5, zorder=4)
ax.axhline(1.0, linestyle='--', color=MUTED, linewidth=1, label='No delay (1.0×)')
ax.set_title('Fig 3  |  Mean Delay Multiplier by Time Slot
(Higher = more delay vs free-flow)', color=TEXT)
ax.set_ylabel('Delay Multiplier (×)', color=TEXT)
ax.set_ylim(0.9, 2.3)
ax.grid(axis='y', zorder=0)
ax.tick_params(axis='x', rotation=20)
ax.set_facecolor(SURFACE)
for spine in ax.spines.values(): spine.set_edgecolor(BORDER)

legend_patches = [
    mpatches.Patch(color=GREEN,  label='Low (<1.2×)'),
    mpatches.Patch(color=ORANGE, label='Moderate (1.2–1.7×)'),
    mpatches.Patch(color=RED,    label='High (>1.7×)'),
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)

for bar, val in zip(bars, delay_slot['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}×', ha='center', va='bottom', color=TEXT, fontsize=9)

plt.tight_layout()
plt.savefig('charts/fig3_delay_multiplier.png')
plt.show()
print("Saved: charts/fig3_delay_multiplier.png")


## 5. Route-Level Performance Comparison

In [ ]:
# Fig 4: Grouped bar — peak AM vs peak PM speed per route
route_peak = traffic[traffic['time_slot'].isin(['peak_am_07_10','peak_pm_17_20'])]
route_pivot = route_peak.pivot(index='route_id', columns='time_slot', values='average_speed_kmh')

# Colour by route type
rt_map = traffic.drop_duplicates('route_id').set_index('route_id')['route_type']
x = np.arange(len(route_pivot))
w = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - w/2, route_pivot['peak_am_07_10'], w, label='Peak AM (07–10)',
               color=[GREEN if rt_map[r]=='BRTS' else ORANGE for r in route_pivot.index],
               alpha=0.85, edgecolor=DARK_BG, linewidth=0.8)
bars2 = ax.bar(x + w/2, route_pivot['peak_pm_17_20'], w, label='Peak PM (17–20)',
               color=[GREEN if rt_map[r]=='BRTS' else RED for r in route_pivot.index],
               alpha=0.75, edgecolor=DARK_BG, linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(route_pivot.index, rotation=30, ha='right')
ax.set_ylabel('Average Speed (km/h)', color=TEXT)
ax.set_title('Fig 4  |  Peak AM vs Peak PM Speed by Route
(Green = BRTS; Orange/Red = Non-BRTS)', color=TEXT)
ax.set_ylim(8, 25)
ax.axhline(14.4, linestyle=':', color=YELLOW, linewidth=1.3, label='TomTom PM baseline 14.4 km/h')
ax.axhline(15.7, linestyle=':', color=MUTED,  linewidth=1.3, label='TomTom AM baseline 15.7 km/h')
ax.legend(fontsize=9)
ax.grid(axis='y', zorder=0)
ax.set_facecolor(SURFACE)
for spine in ax.spines.values(): spine.set_edgecolor(BORDER)

plt.tight_layout()
plt.savefig('charts/fig4_route_peak_speeds.png')
plt.show()
print("Saved: charts/fig4_route_peak_speeds.png")


## 6. Incident Type & Severity

In [ ]:
# Fig 5: Donut — incident type distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Fig 5  |  Incident Distribution & Severity Split', color=TEXT, fontsize=13)

inc_counts = incidents['incident_type'].value_counts()
palette = [RED, ORANGE, YELLOW, ACCENT, GREEN, '#b39ddb']

wedges, texts, autotexts = axes[0].pie(
    inc_counts.values, labels=inc_counts.index,
    autopct='%1.1f%%', startangle=90,
    colors=palette[:len(inc_counts)],
    pctdistance=0.75, labeldistance=1.12,
    wedgeprops={'edgecolor': DARK_BG, 'linewidth': 2}
)
for t in texts: t.set_color(TEXT)
for at in autotexts: at.set_color(DARK_BG); at.set_fontsize(8)

# Draw inner circle for donut
centre_circle = plt.Circle((0,0), 0.55, fc=SURFACE)
axes[0].add_artist(centre_circle)
axes[0].set_title('Incident Type Distribution', color=TEXT)

# Stacked bar — severity by type
sev_cross = incidents.groupby(['incident_type','severity']).size().unstack(fill_value=0)
sev_cross = sev_cross.loc[inc_counts.index]  # same order

bottom = np.zeros(len(sev_cross))
colors_sev = {c: col for c, col in zip(sev_cross.columns, [GREEN, RED, ORANGE])}
for col in sev_cross.columns:
    axes[1].barh(sev_cross.index, sev_cross[col], left=bottom,
                 label=col, color=colors_sev.get(col, ACCENT), alpha=0.85, edgecolor=DARK_BG)
    bottom += sev_cross[col].values
axes[1].set_xlabel('Number of Incidents', color=TEXT)
axes[1].set_title('Severity Split by Incident Type', color=TEXT)
axes[1].legend(fontsize=9)
axes[1].set_facecolor(SURFACE)
for spine in axes[1].spines.values(): spine.set_edgecolor(BORDER)
axes[0].set_facecolor(SURFACE)

plt.tight_layout()
plt.savefig('charts/fig5_incident_distribution.png')
plt.show()
print("Saved: charts/fig5_incident_distribution.png")


In [ ]:
# Fig 6: Duration box plots by incident type
fig, ax = plt.subplots(figsize=(11, 5))
inc_order = incidents.groupby('incident_type')['duration_minutes'].median().sort_values(ascending=False).index

data_groups = [incidents[incidents['incident_type']==t]['duration_minutes'].values for t in inc_order]
bp = ax.boxplot(data_groups, labels=inc_order, patch_artist=True,
                medianprops={'color': TEXT, 'linewidth': 2},
                whiskerprops={'color': MUTED}, capprops={'color': MUTED},
                flierprops={'marker': 'o', 'color': RED, 'markersize': 5, 'alpha': 0.5},
                boxprops={'edgecolor': BORDER})

for patch, color in zip(bp['boxes'], palette[:len(inc_order)]):
    patch.set_facecolor(color); patch.set_alpha(0.7)

ax.set_ylabel('Duration (minutes)', color=TEXT)
ax.set_title('Fig 6  |  Incident Duration Distribution by Type', color=TEXT)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', zorder=0)
ax.set_facecolor(SURFACE)
for spine in ax.spines.values(): spine.set_edgecolor(BORDER)

plt.tight_layout()
plt.savefig('charts/fig6_incident_duration.png')
plt.show()
print("Saved: charts/fig6_incident_duration.png")


## 7. Incident Peak-Hour Timing

In [ ]:
# Fig 7: Hourly incident histogram with peak shading
incidents['start_hour'] = pd.to_datetime(incidents['start_time'], format='%H:%M').dt.hour
hour_counts = incidents['start_hour'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
colors_hr = [RED if (7<=h<=9) or (17<=h<=19) else ACCENT for h in hour_counts.index]
bars = ax.bar(hour_counts.index, hour_counts.values, color=colors_hr,
              width=0.7, edgecolor=DARK_BG, linewidth=0.8, zorder=3)

ax.axvspan(6.5, 9.5, alpha=0.08, color=RED, label='AM Peak (07–10)')
ax.axvspan(16.5, 19.5, alpha=0.08, color=ORANGE, label='PM Peak (17–20)')
ax.set_xlabel('Hour of Day', color=TEXT)
ax.set_ylabel('Incident Count', color=TEXT)
ax.set_title('Fig 7  |  Incident Start Time Distribution
(Peak hours highlighted)', color=TEXT)
ax.set_xticks(range(int(hour_counts.index.min()), int(hour_counts.index.max())+1))
ax.legend(fontsize=9)
ax.grid(axis='y', zorder=0)
ax.set_facecolor(SURFACE)
for spine in ax.spines.values(): spine.set_edgecolor(BORDER)

plt.tight_layout()
plt.savefig('charts/fig7_incident_timing.png')
plt.show()
print("Saved: charts/fig7_incident_timing.png")


## 8. Zone & Equity Visualizations

In [ ]:
# Fig 8: Waiting time by zone coloured by income group
if 'est_wait_peak_min' in zones.columns:
    zones_sorted = zones.sort_values('est_wait_peak_min', ascending=True)
    inc_palette  = {'High': GREEN, 'Medium': ORANGE, 'Low': RED}
    bar_colors   = [inc_palette[g] for g in zones_sorted['income_group']]

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(zones_sorted['ward_name'], zones_sorted['est_wait_peak_min'],
                   color=bar_colors, edgecolor=DARK_BG, linewidth=0.8, alpha=0.88)

    for bar, val in zip(bars, zones_sorted['est_wait_peak_min']):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f} min', va='center', ha='left', color=TEXT, fontsize=8.5)

    ax.set_xlabel('Estimated Peak Waiting Time (minutes)', color=TEXT)
    ax.set_title('Fig 8  |  Estimated Peak Waiting Time by Zone
(Coloured by income group)', color=TEXT)
    ax.grid(axis='x', zorder=0)
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values(): spine.set_edgecolor(BORDER)

    legend_patches = [mpatches.Patch(color=c, label=g) for g, c in inc_palette.items()]
    ax.legend(handles=legend_patches, title='Income Group', fontsize=9, loc='lower right')

    plt.tight_layout()
    plt.savefig('charts/fig8_waiting_time_by_zone.png')
    plt.show()
    print("Saved: charts/fig8_waiting_time_by_zone.png")
else:
    print("est_wait_peak_min not available — run the analysis notebook first.")


In [ ]:
# Fig 9: Transport dependency vs reliability_score scatter
if all(c in zones.columns for c in ['transport_dependency','reliability_score']):
    inc_palette = {'High': GREEN, 'Medium': ORANGE, 'Low': RED}
    fig, ax = plt.subplots(figsize=(9, 6))

    for group, gdf in zones.groupby('income_group'):
        ax.scatter(gdf['transport_dependency'], gdf['reliability_score'],
                   c=inc_palette[group], s=gdf['population']/1000 + 40,
                   alpha=0.85, edgecolors=DARK_BG, linewidths=0.8, label=group+' Income', zorder=4)
        for _, row in gdf.iterrows():
            ax.annotate(row['ward_name'], (row['transport_dependency'], row['reliability_score']),
                        fontsize=7, color=MUTED, xytext=(5,3), textcoords='offset points')

    # Regression line
    r, p = stats.pearsonr(zones['transport_dependency'], zones['reliability_score'])
    m, b = np.polyfit(zones['transport_dependency'], zones['reliability_score'], 1)
    xs = np.linspace(zones['transport_dependency'].min(), zones['transport_dependency'].max(), 100)
    ax.plot(xs, m*xs + b, '--', color=TEXT, linewidth=1.5, alpha=0.6,
            label=f'Linear fit (r={r:.2f}, p={p:.3f})')

    ax.set_xlabel('Transport Dependency Score (0–1)', color=TEXT)
    ax.set_ylabel('Reliability Score (0–100)', color=TEXT)
    ax.set_title('Fig 9  |  Transport Dependency vs Reliability by Zone
(Bubble size ∝ population)', color=TEXT)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(zorder=0)
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values(): spine.set_edgecolor(BORDER)

    plt.tight_layout()
    plt.savefig('charts/fig9_dependency_reliability.png')
    plt.show()
    print(f"Saved: charts/fig9_dependency_reliability.png  (r={r:.3f})")
else:
    print("Columns missing — check zones.csv")


In [ ]:
# Fig 10: Income group comparison — grouped bars (mean wait, mean reliability, mean dependency)
if all(c in zones.columns for c in ['transport_dependency','reliability_score','est_wait_peak_min']):
    inc_order = ['High','Medium','Low']
    metrics = {
        'Avg Wait (min)': zones.groupby('income_group')['est_wait_peak_min'].mean(),
        'Reliability (0–100)': zones.groupby('income_group')['reliability_score'].mean(),
        'Dependency (×10)': zones.groupby('income_group')['transport_dependency'].mean() * 10,
    }

    x = np.arange(len(inc_order))
    w = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))

    colors_bar = [GREEN, ACCENT, ORANGE]
    for i, (label, vals) in enumerate(metrics.items()):
        bars = ax.bar(x + i*w - w, [vals[g] for g in inc_order], w,
                      label=label, color=colors_bar[i], alpha=0.85, edgecolor=DARK_BG)
        for bar, val in zip(bars, [vals[g] for g in inc_order]):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=8, color=TEXT)

    ax.set_xticks(x)
    ax.set_xticklabels(inc_order)
    ax.set_title('Fig 10  |  Key Equity Metrics by Income Group', color=TEXT)
    ax.legend(fontsize=9)
    ax.grid(axis='y', zorder=0)
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values(): spine.set_edgecolor(BORDER)

    plt.tight_layout()
    plt.savefig('charts/fig10_equity_comparison.png')
    plt.show()
    print("Saved: charts/fig10_equity_comparison.png")


## 9. GTFS Network Visualisation

In [ ]:
# Fig 11: Stop locations map (scatter plot as proxy)
if all(c in stops.columns for c in ['stop_lat','stop_lon']):
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.scatter(stops['stop_lon'], stops['stop_lat'], s=12, color=ACCENT,
               alpha=0.5, edgecolors='none', zorder=4)
    ax.set_xlabel('Longitude', color=TEXT)
    ax.set_ylabel('Latitude', color=TEXT)
    ax.set_title('Fig 11  |  PMPML Stop Locations (WGS84)
Pune, Maharashtra', color=TEXT)
    ax.grid(zorder=0, alpha=0.3)
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values(): spine.set_edgecolor(BORDER)
    ax.text(0.02, 0.02, f'n = {len(stops)} stops', transform=ax.transAxes,
            color=MUTED, fontsize=9)
    plt.tight_layout()
    plt.savefig('charts/fig11_stop_locations.png')
    plt.show()
    print("Saved: charts/fig11_stop_locations.png")
else:
    print("stop_lat / stop_lon columns not found — check stops.txt")


In [ ]:
# Fig 12: Trips per route (top 20)
trips_per_route = trips.groupby('route_id').size().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(trips_per_route.index, trips_per_route.values,
              color=ACCENT, alpha=0.82, edgecolor=DARK_BG, linewidth=0.8)
ax.set_xlabel('Route ID', color=TEXT)
ax.set_ylabel('Number of Trips', color=TEXT)
ax.set_title('Fig 12  |  Top 20 Routes by Trip Frequency', color=TEXT)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', zorder=0)
ax.set_facecolor(SURFACE)
for spine in ax.spines.values(): spine.set_edgecolor(BORDER)

plt.tight_layout()
plt.savefig('charts/fig12_trips_per_route.png')
plt.show()
print("Saved: charts/fig12_trips_per_route.png")


## 10. Correlation Matrix

In [ ]:
# Fig 13: Correlation matrix — traffic numerical columns
num_cols = ['congestion_level','average_speed_kmh','delay_multiplier']
corr = traffic[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.zeros_like(corr, dtype=bool)

sns.heatmap(corr, ax=ax, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, linewidths=1, linecolor=DARK_BG,
            annot_kws={'size': 12, 'color': TEXT},
            cbar_kws={'shrink': 0.8})
ax.set_title('Fig 13  |  Traffic Metrics Correlation Matrix', color=TEXT, pad=14)
ax.set_xticklabels(ax.get_xticklabels(), color=TEXT, rotation=20)
ax.set_yticklabels(ax.get_yticklabels(), color=TEXT, rotation=0)
ax.collections[0].colorbar.ax.yaxis.label.set_color(TEXT)
ax.collections[0].colorbar.ax.tick_params(colors=TEXT)

plt.tight_layout()
plt.savefig('charts/fig13_correlation_matrix.png')
plt.show()
print("Saved: charts/fig13_correlation_matrix.png")


## 11. Chart Summary

In [ ]:
saved = [f for f in os.listdir('charts') if f.endswith('.png')]
saved.sort()
print(f"{'='*50}")
print(f"  {len(saved)} charts saved to ./charts/")
print(f"{'='*50}")
for f in saved:
    path = f'charts/{f}'
    size = os.path.getsize(path) // 1024
    print(f"  {f:<45} {size:>5} KB")
print(f"{'='*50}")
print("\nAll charts are IEEE-paper ready at 150 DPI.")
print("Reference them in your paper as Figures 1–13.")


---
*Generated for EquiTwin (Pune/PMPML) — Member 4 | Data Visualization Notebook*  
*Python 3.10 | pandas | numpy | matplotlib | seaborn | Random Seed 42*  
*Charts saved to ./charts/ directory*
